# Frozen Euclidean-MSE adapter: change the goal

Select **GPU → Run all**. This loads the existing **distance-fitted adapter**, freezes it and evaluates a new goal. **No training, optimizer, collection or recalibration occurs.** Existing experiments remain unchanged.

Defaults: final **stage4, radius34**, original goal **(50,42)**, new goal **(46,38)**. Edit `SOURCE_RUN`, `SOURCE_STAGE` or `NEW_GOAL` below if desired.

Main comparison uses the same ball positions for both goals, inside the original fitting disk and within its trained distance range relative to both goals. Heatmaps show the full right compartment, including outside probes.

The encoder images contain a single ball and walls, with no visible goal marker. Moving the goal means using a fresh reference image with the ball at the new target position. The adapter is applied symmetrically to observation and reference tokens as before.

Distance estimate stays **34 × adapted MSE**. The raw baseline uses the original training scalar unchanged. The original-goal control helps distinguish goal transfer from fresh-encoding/runtime differences.


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, shutil, signal, subprocess, sys, time
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime, then Run all.'
DRIVE = Path('/content/drive/MyDrive')
SOURCE_RUN = DRIVE / 'datasets/curriculum_cost_adapter/radial_distance_v1/20261106T101715235909Z'
SOURCE_STAGE = 4
NEW_GOAL = [46.0, 38.0]
assert (SOURCE_RUN/'run.json').is_file(), 'Set SOURCE_RUN to your existing radial_distance_v1 run folder.'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = DRIVE / 'datasets/curriculum_cost_adapter/radial_goal_transfer_v1' / RUN_ID
REPO = Path('/content/i-jepa-wm-radial-goal-transfer')
REPO_URL = 'https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_COMMIT = '2cdafe088d46b4843a5147928c049025e3ce33fe'
DINO_ROOT = Path('/content/dino_wm')
DINO_COMMIT = '0a9492fa12044b852ae9e001cc74604b79c8bb0c'
ENCODER_DRIVE = DRIVE / 'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
ENCODER = Path('/content/radial-goal-transfer-artifacts/encoder.pth.tar')
ENCODER_SHA = '3f37d8a7572e25c2dce9a3072c2fd8cd55ce32dc6edb92a8996b0e4cb140bf98'
assert ENCODER_DRIVE.is_file(), f'Missing encoder: {ENCODER_DRIVE}'

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8*1024*1024), b''): h.update(block)
    return h.hexdigest()

def shell(command, cwd=None):
    command=list(map(str,command)); print('$',' '.join(command),flush=True)
    with subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        try:
            for line in process.stdout: print(line,end='',flush=True)
            returncode=process.wait()
        except BaseException:
            process.send_signal(signal.SIGINT)
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.terminate()
            raise
    if returncode: raise subprocess.CalledProcessError(returncode,command)

print('GPU:', torch.cuda.get_device_name())
print('New Drive results:', OUTPUT)
print('Frozen source run:',SOURCE_RUN,'stage',SOURCE_STAGE)
print('New goal:',NEW_GOAL)


## Pinned diagnostic and frozen encoder

Only the existing source Drive run and encoder checkpoint are needed. The previous local token cache is not required. Source input/checkpoint hashes and the entire completed checkpoint chain are validated. The historical fitting-code fingerprint is preserved as provenance; it is expected to differ from this newer diagnostic implementation.


In [ ]:
if not REPO.exists():
    shell(['git','clone','--branch','curriculum-cost-adapter','--single-branch',REPO_URL,REPO])
else:
    assert subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip()==REPO_URL
    assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip(), 'Clone has local changes.'
    shell(['git','-C',REPO,'fetch','origin','curriculum-cost-adapter'])
assert SOURCE_COMMIT, 'Use the published notebook with a verified source pin.'
shell(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT])
shell([sys.executable,'-m','pip','install','-q','gym==0.26.2','omegaconf','pyyaml','scipy','einops','matplotlib','pillow'])
if not DINO_ROOT.exists(): shell(['git','clone','https://github.com/gaoyuezhou/dino_wm.git',DINO_ROOT])
assert not subprocess.check_output(['git','-C',str(DINO_ROOT),'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Renderer has tracked changes.'
shell(['git','-C',DINO_ROOT,'checkout',DINO_COMMIT])
ENCODER.parent.mkdir(parents=True,exist_ok=True)
if not ENCODER.exists() or sha256(ENCODER)!=ENCODER_SHA: shutil.copy2(ENCODER_DRIVE,ENCODER)
assert sha256(ENCODER)==ENCODER_SHA, 'Unexpected encoder weights.'
print('Ready. The saved distance-adapter run supplies the frozen weights; no training will run.')

def command(operation):
    args=[sys.executable,'-m','src.mylonas_ijepa_wm.radial_goal_transfer',operation,
          '--source-run',SOURCE_RUN,'--output-dir',OUTPUT,'--stage',SOURCE_STAGE,
          '--new-goal',*NEW_GOAL]
    if operation=='evaluate': args+=['--encoder-checkpoint',ENCODER,'--dino-wm-root',DINO_ROOT,'--device','cuda']
    return args


## Evaluate both goals with unchanged weights

Freshly encode the full-right grid (406positions) and goal coordinates in batches of eight. Each coordinate is encoded once, and the reference tokens are reused for self-comparisons. The raw self-cost is therefore exactly zero.

The renderer verifies that changing the environment goal does not alter fixed-position encoder images. The saved adapter parameters and source checkpoint files must remain unchanged. The diagnostic writes only its own new timestamped `radial_goal_transfer_v1` output.

Rerunning this cell reuses validated completed results under the same settings. Changing setup creates a fresh timestamped run.


In [ ]:
started=time.monotonic()
shell(command('evaluate'),cwd=REPO)
print(f'Frozen goal-transfer evaluation took {(time.monotonic()-started)/60:.2f} minutes.')


## Read the goal-transfer results

- **Heldout common** is the main comparison: same grid positions for both goals, excluding coordinate/image overlaps with original fitting inputs and the two reference images. Both goals stay within the original fitting region; tested distances stay within the fitted range.
- **Common** also includes fitting positions. **Local** uses common positions within6units of the evaluated goal; the local populations differ by goal.
- **Full domain** includes extrapolation outside the original region/range and should be interpreted separately.
- Distance MAE/RMSE are simulation units. Raw calibration remains the scalar fitted for the original goal; it is never re-fitted here.
- Directional tests compare adjacent grid points oriented toward each evaluated goal by their true distance. Equal-distance edges are omitted. Uphill means the cost increases while moving closer. Ties and empty denominators are retained.
- White map points mark original fitting inputs; the dashed circle remains centered at the **original** fitting region. Black x is the original goal; orange star is the evaluated goal.
- Adapted/true cost maps share a full-range scale; raw/error maps have separate scales. No values are clipped.


In [ ]:
shell(command('report'),cwd=REPO)  # Validate saved report/source lineage before displaying it.
r=json.loads((OUTPUT/'report.json').read_text())
from IPython.display import display,Image

def number(value): return 'N/A' if value is None else f'{value:.4f}'
def pct(value): return 'N/A' if value is None else f'{value:.1%}'

print('Source stage/radius:',r['source']['stage'],r['source']['radius'])
print('Adapter unchanged:',r['adapter_unchanged'],'| distance scale:',r['distance_scale'])
print('Original training raw scalar, reused unchanged:',r['raw_calibration'])
print('Original-goal cached/fresh consistency:',r['original_goal_consistency'])
print('If the original-goal control changes substantially, interpret transfer alongside that runtime/reference difference.')
for j,g in enumerate(r['goals']):
    print('\n','ORIGINAL-GOAL CONTROL' if j==0 else 'CHANGED GOAL',g['goal'])
    print('Reference image: ball at the evaluated goal; no goal marker is encoded.')
    display(Image(filename=str(OUTPUT/f'goal_{j}.png')))
    print('Raw/adapted self-cost:',g['raw_self_cost'],g['adapted_self_cost'])
    print(f"{'cohort':>20} {'positions':>10} {'method':>17} {'distance MAE':>14} {'distance RMSE':>14}")
    for cohort in ('heldout_common','common','local','full_domain'):
        m=g[cohort]
        for method in ('raw_calibrated','adapted'):
            print(f"{cohort:>20} {m['count']:>10} {method:>17} {number(m[method]['mae']):>14} {number(m[method]['rmse']):>14}")
    for key in ('directional','heldout_directional'):
        m=g[key]
        print(key,'| comparisons:',m['comparisons'],
              '| raw uphill:',pct(m['raw_up_fraction']),'ties:',m['raw_ties'],
              '| adapted uphill:',pct(m['adapted_up_fraction']),'ties:',m['adapted_ties'])
    display(Image(filename=str(OUTPUT/f'goal_{j}_heatmaps.png')))
    display(Image(filename=str(OUTPUT/f'goal_{j}_calibration.png')))


## Compare the same heldout ball positions

The original-goal row is the control; the changed-goal row tests transfer without fitting. Similar errors and a cost basin centered at the new goal support transfer for this shift. A good original-goal control with a degraded changed-goal map indicates goal-specific fitting. Neither outcome establishes arbitrary-goal or planning transfer.


In [ ]:
print('MAIN COMPARISON: identical heldout common-region grid positions')
print(f"{'goal':>15} {'positions':>10} {'raw MAE':>12} {'adapt MAE':>12} {'adapt RMSE':>12} {'adapt uphill':>14}")
for g in r['goals']:
    m=g['heldout_common'];edges=g['heldout_directional']
    print(f"{str(tuple(g['goal'])):>15} {m['count']:>10} {number(m['raw_calibrated']['mae']):>12} "
          f"{number(m['adapted']['mae']):>12} {number(m['adapted']['rmse']):>12} {pct(edges['adapted_up_fraction']):>14}")
print('No optimization or new-goal calibration was performed.')
print('Full-right maps include outside-region points; main table excludes them.')
print('Results:',OUTPUT)


## Return the executed notebook

Download **File → Download → Download .ipynb** and attach it here. Fresh goal references, numeric scores, source provenance, reports and four PNG/PDF figures are saved in the new Drive output. The source adapter experiment is unchanged.

Local synthetic tests verify the diagnostic workflow. Actual GPU results determine whether this frozen adapter transfers to the selected goal shift.
